In [ ]:
# ============================================================
# CELL 18 — MS1MV2: 30,000-person enrollment protocol
# ============================================================

from pathlib import Path
import numpy as np
import random

SEED = 42

NUM_ENROLLMENT_IDENTITIES = 30000
NUM_UNENROLLED_IDENTITIES = 302
TOTAL_REQUIRED_IDENTITIES = (
    NUM_ENROLLMENT_IDENTITIES
    + NUM_UNENROLLED_IDENTITIES
)

random.seed(SEED)
rng = np.random.default_rng(SEED)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}


# ------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------

MS1_ROOT = Path(
    "/kaggle/input/datasets/yakhyokhuja/ms1m-arcface-dataset/ms1m-arcface"
)

if not MS1_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset root not found:\n{MS1_ROOT}\n\n"
        "If Kaggle uses a slightly different path, "
        "copy the path shown in the right-side file browser."
    )


print("=" * 80)
print("MS1MV2 — 30,000-IDENTITY WATCHLIST")
print("=" * 80)

print("Dataset root:")
print(MS1_ROOT)


# ------------------------------------------------------------
# Get identity directories only
# ------------------------------------------------------------

identity_dirs = [
    p
    for p in MS1_ROOT.iterdir()
    if p.is_dir()
]


print(
    f"\nIdentity directories found: "
    f"{len(identity_dirs):,}"
)


if len(identity_dirs) < TOTAL_REQUIRED_IDENTITIES:
    raise RuntimeError(
        f"Need at least {TOTAL_REQUIRED_IDENTITIES:,} "
        f"identity folders, but found only "
        f"{len(identity_dirs):,}."
    )


# ------------------------------------------------------------
# Shuffle identities reproducibly
# ------------------------------------------------------------

identity_dirs = sorted(
    identity_dirs,
    key=lambda p: p.name
)

rng.shuffle(identity_dirs)


# ------------------------------------------------------------
# Find first usable image from each identity
#
# We stop as soon as we have 30,302 identities.
# Therefore we do NOT scan the whole dataset recursively.
# ------------------------------------------------------------

selected_records = []


for identity_dir in identity_dirs:

    first_image = None

    for file_path in identity_dir.iterdir():

        if (
            file_path.is_file()
            and
            file_path.suffix.lower()
            in IMAGE_EXTENSIONS
        ):
            first_image = file_path
            break

    if first_image is None:
        continue

    selected_records.append(
        (
            identity_dir.name,
            first_image
        )
    )

    if (
        len(selected_records)
        >= TOTAL_REQUIRED_IDENTITIES
    ):
        break


if len(selected_records) < TOTAL_REQUIRED_IDENTITIES:
    raise RuntimeError(
        f"Could only find "
        f"{len(selected_records):,} usable identities."
    )


# ------------------------------------------------------------
# Split identities
# ------------------------------------------------------------

enrollment_records = selected_records[
    :NUM_ENROLLMENT_IDENTITIES
]

unenrolled_records = selected_records[
    NUM_ENROLLMENT_IDENTITIES:
    NUM_ENROLLMENT_IDENTITIES
    + NUM_UNENROLLED_IDENTITIES
]


# ------------------------------------------------------------
# Enrollment
# ------------------------------------------------------------

enrollment_identity_labels = [
    identity
    for identity, path
    in enrollment_records
]

enrollment_image_paths = [
    path
    for identity, path
    in enrollment_records
]


# ------------------------------------------------------------
# Unenrolled
# ------------------------------------------------------------

unenrolled_query_labels = [
    identity
    for identity, path
    in unenrolled_records
]

unenrolled_query_paths = [
    path
    for identity, path
    in unenrolled_records
]


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(
    enrollment_image_paths
) == 30000

assert len(
    set(enrollment_identity_labels)
) == 30000

assert len(
    unenrolled_query_paths
) == 302

assert len(
    set(unenrolled_query_labels)
) == 302

assert set(
    enrollment_identity_labels
).isdisjoint(
    set(unenrolled_query_labels)
)


print("\n" + "=" * 80)
print("PROTOCOL SUMMARY")
print("=" * 80)

print(
    f"Enrollment identities : "
    f"{len(enrollment_identity_labels):,}"
)

print(
    f"Enrollment images     : "
    f"{len(enrollment_image_paths):,}"
)

print(
    f"Images per identity   : 1"
)

print(
    f"Held-out identities   : "
    f"{len(unenrolled_query_labels):,}"
)

print(
    f"Held-out images       : "
    f"{len(unenrolled_query_paths):,}"
)


print("\nFirst 5 enrollment samples:")

for i in range(5):
    print(
        f"{i:2d} | "
        f"ID {enrollment_identity_labels[i]:<10} | "
        f"{enrollment_image_paths[i].name}"
    )


print("\nFirst 5 unenrolled samples:")

for i in range(5):
    print(
        f"{i:2d} | "
        f"ID {unenrolled_query_labels[i]:<10} | "
        f"{unenrolled_query_paths[i].name}"
    )


print(
    "\n✓ 30,000-person enrollment watchlist created."
)

print(
    "✓ 302 completely unseen identities reserved."
)

In [ ]:
# ============================================================
# CELL 19 — Select 302 exact enrolled queries
# ============================================================

NUM_GENUINE_QUERIES = 302

selected_enrollment_indices = rng.choice(
    NUM_ENROLLMENT_IDENTITIES,
    size=NUM_GENUINE_QUERIES,
    replace=False
)


exact_query_paths = [
    enrollment_image_paths[int(i)]
    for i in selected_enrollment_indices
]

exact_query_labels = [
    enrollment_identity_labels[int(i)]
    for i in selected_enrollment_indices
]


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(exact_query_paths) == 302

assert len(
    set(exact_query_labels)
) == 302

assert len(
    set(exact_query_paths)
) == 302


print("=" * 80)
print("EXACT ENROLLED QUERY SET")
print("=" * 80)

print(
    f"Exact queries      : "
    f"{len(exact_query_paths)}"
)

print(
    f"Unique identities  : "
    f"{len(set(exact_query_labels))}"
)

print(
    f"Unique images      : "
    f"{len(set(exact_query_paths))}"
)


print("\nFirst 10:")

for i in range(10):

    print(
        f"{i:3d} | "
        f"ID {exact_query_labels[i]:<10} | "
        f"{exact_query_paths[i].name}"
    )


print(
    "\n✓ 302 exact enrolled queries selected."
)

In [ ]:
# ============================================================
# CELL 20 — Generate 302 paired 1%-noise queries
# ============================================================

from PIL import Image
from pathlib import Path
import numpy as np

NOISE_RATE = 0.01

NOISY_DIR = Path(
    "/kaggle/working/ms1mv2_noisy_queries_1pct"
)

NOISY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

noisy_query_paths = []
noisy_query_labels = []

actual_noise_rates = []


for i, (image_path, identity) in enumerate(
    zip(
        exact_query_paths,
        exact_query_labels
    )
):

    # --------------------------------------------------------
    # Load original image
    # --------------------------------------------------------

    image = Image.open(
        image_path
    ).convert("RGB")

    original = np.asarray(
        image,
        dtype=np.uint8
    )

    flat = original.reshape(-1)

    # --------------------------------------------------------
    # Exactly 1% of RGB channel values will be changed
    # --------------------------------------------------------

    n_corrupt = int(
        round(
            flat.size * NOISE_RATE
        )
    )

    local_rng = np.random.default_rng(
        SEED + 10000 + i
    )

    positions = local_rng.choice(
        flat.size,
        size=n_corrupt,
        replace=False
    )

    noisy_flat = flat.copy()

    # Random non-zero offset guarantees that every selected
    # channel value actually changes.
    offsets = local_rng.integers(
        1,
        256,
        size=n_corrupt,
        dtype=np.uint16
    )

    original_values = (
        noisy_flat[positions]
        .astype(np.uint16)
    )

    changed_values = (
        original_values + offsets
    ) % 256

    noisy_flat[positions] = (
        changed_values.astype(
            np.uint8
        )
    )

    noisy = noisy_flat.reshape(
        original.shape
    )


    # --------------------------------------------------------
    # Verify actual corruption
    # --------------------------------------------------------

    changed = np.count_nonzero(
        original != noisy
    )

    actual_rate = (
        changed / original.size
    )

    actual_noise_rates.append(
        actual_rate
    )


    # --------------------------------------------------------
    # Save losslessly
    # --------------------------------------------------------

    output_path = (
        NOISY_DIR
        /
        f"noisy_{i:03d}_id_{identity}.png"
    )

    Image.fromarray(
        noisy
    ).save(
        output_path
    )

    noisy_query_paths.append(
        output_path
    )

    noisy_query_labels.append(
        identity
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

actual_noise_rates = np.asarray(
    actual_noise_rates
)

assert len(noisy_query_paths) == 302

assert (
    noisy_query_labels
    ==
    exact_query_labels
)


print("=" * 80)
print("1% NOISY QUERY SET")
print("=" * 80)

print(
    f"Noisy queries       : "
    f"{len(noisy_query_paths)}"
)

print(
    f"Target noise        : "
    f"{NOISE_RATE * 100:.2f}%"
)

print(
    f"Minimum actual      : "
    f"{actual_noise_rates.min() * 100:.4f}%"
)

print(
    f"Maximum actual      : "
    f"{actual_noise_rates.max() * 100:.4f}%"
)

print(
    f"Mean actual         : "
    f"{actual_noise_rates.mean() * 100:.4f}%"
)

print(
    f"Saved directory     : "
    f"{NOISY_DIR}"
)

print(
    "\n✓ 302 paired noisy queries created."
)

In [ ]:
# ============================================================
# CELL 21 — Verify enrollment/query protocol
# ============================================================

enrollment_id_set = set(
    enrollment_identity_labels
)

exact_id_set = set(
    exact_query_labels
)

noisy_id_set = set(
    noisy_query_labels
)

unenrolled_id_set = set(
    unenrolled_query_labels
)


print("=" * 80)
print("MS1MV2 PROTOCOL VALIDATION")
print("=" * 80)

print(
    f"Enrollment identities       : "
    f"{len(enrollment_id_set):,}"
)

print(
    f"Exact query identities      : "
    f"{len(exact_id_set):,}"
)

print(
    f"Noisy query identities      : "
    f"{len(noisy_id_set):,}"
)

print(
    f"Unenrolled query identities : "
    f"{len(unenrolled_id_set):,}"
)


print("\nIdentity overlap:")

print(
    "Enrollment ∩ Exact     :",
    len(
        enrollment_id_set
        &
        exact_id_set
    )
)

print(
    "Enrollment ∩ Noisy     :",
    len(
        enrollment_id_set
        &
        noisy_id_set
    )
)

print(
    "Enrollment ∩ Unenrolled:",
    len(
        enrollment_id_set
        &
        unenrolled_id_set
    )
)


# ------------------------------------------------------------
# Expected conditions
# ------------------------------------------------------------

assert len(
    enrollment_id_set
    &
    exact_id_set
) == 302

assert len(
    enrollment_id_set
    &
    noisy_id_set
) == 302

assert len(
    enrollment_id_set
    &
    unenrolled_id_set
) == 0

assert exact_query_labels == noisy_query_labels


print(
    "\n✓ Exact queries are enrolled."
)

print(
    "✓ Noisy queries correspond to the same enrolled identities."
)

print(
    "✓ All 302 non-member identities are absent from enrollment."
)

In [ ]:
# ============================================================
# CELL 22 — Assemble final 906-query protocol
# ============================================================

query_image_paths = (
    exact_query_paths
    +
    noisy_query_paths
    +
    unenrolled_query_paths
)

query_identity_labels = (
    exact_query_labels
    +
    noisy_query_labels
    +
    unenrolled_query_labels
)

query_group_labels = (
    ["Exact Enrolled"] * 302
    +
    ["Noisy Enrolled (1%)"] * 302
    +
    ["Unenrolled"] * 302
)


# 1 = should be recognized as a watchlist member
# 0 = should be rejected as a non-member

y_true_images = np.asarray(
    [1] * 302
    +
    [1] * 302
    +
    [0] * 302,
    dtype=np.int32
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(
    query_image_paths
) == 906

assert len(
    query_identity_labels
) == 906

assert len(
    query_group_labels
) == 906

assert len(
    y_true_images
) == 906


print("=" * 80)
print("FINAL MS1MV2 WATCHLIST PROTOCOL")
print("=" * 80)

print(
    f"Enrollment identities/images : "
    f"{len(enrollment_image_paths):,}"
)

print("\nQuery set:")

print(
    "  Exact enrolled             : 302"
)

print(
    "  1% noisy enrolled          : 302"
)

print(
    "  Unenrolled identities      : 302"
)

print(
    "  --------------------------------"
)

print(
    "  Total                      : 906"
)

print("\nGround truth:")

print(
    f"  Members     : "
    f"{int(y_true_images.sum())}"
)

print(
    f"  Non-members : "
    f"{int((y_true_images == 0).sum())}"
)


print(
    "\n✓ 30,000 enrollment + 906-query "
    "protocol is ready."
)

In [ ]:
import time
import numpy as np
from datetime import datetime

DURATION_HOURS = 1.5
DURATION_SECONDS = DURATION_HOURS * 60 * 60

start = time.time()
end = start + DURATION_SECONDS

iterations = 0

print("Computation started:", datetime.now())
print("Target duration: 1.5 hours")

while time.time() < end:
    # Small but real CPU computation
    a = np.random.rand(500, 500)
    b = np.random.rand(500, 500)
    c = a @ b

    iterations += 1

    # Print progress roughly every 50 iterations
    if iterations % 50 == 0:
        elapsed = time.time() - start
        remaining = max(0, end - time.time())

        print(
            f"Elapsed: {elapsed/60:.1f} min | "
            f"Remaining: {remaining/60:.1f} min | "
            f"Iterations: {iterations}",
            flush=True
        )

print("\nFinished.")
print("Total runtime:", (time.time() - start) / 60, "minutes")
print("Total iterations:", iterations)

In [ ]:
# ============================================================
# CELL 23 — Install ArcFace dependencies
# ============================================================

import sys
import subprocess
import importlib.util


def ensure_package(import_name, pip_name):

    if importlib.util.find_spec(import_name) is None:

        print(f"Installing {pip_name}...")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            pip_name
        ])

    else:

        print(
            f"{import_name} already installed."
        )


ensure_package(
    "insightface",
    "insightface"
)

ensure_package(
    "onnxruntime",
    "onnxruntime-gpu"
)


print(
    "\n✓ ArcFace dependencies ready."
)

In [ ]:
# ============================================================
# CELL 24 — Check ONNX Runtime providers
# ============================================================

import onnxruntime as ort
import torch

print("=" * 80)
print("RUNTIME CHECK")
print("=" * 80)

print(
    "PyTorch CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


print(
    "\nONNX Runtime providers:"
)

for provider in ort.get_available_providers():
    print(
        " ",
        provider
    )

In [ ]:
# ============================================================
# CELL 25 — Replace CPU ONNX Runtime with GPU ONNX Runtime
# ============================================================

import sys
import subprocess

print("Removing CPU/GPU ONNX Runtime packages if present...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "onnxruntime",
        "onnxruntime-gpu"
    ],
    check=False
)

print("\nInstalling GPU ONNX Runtime...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "onnxruntime-gpu"
    ]
)

print("\n✓ onnxruntime-gpu installed.")
print(
    "\nIMPORTANT: Restart the Kaggle session/kernel now "
    "before running the next cell."
)

In [ ]:
# ============================================================
# CELL 24B — Save current MS1MV2 protocol
# ============================================================

import pickle
from pathlib import Path

PROTOCOL_FILE = Path(
    "/kaggle/working/ms1mv2_hbf_protocol.pkl"
)

protocol = {
    "enrollment_image_paths": [
        str(p) for p in enrollment_image_paths
    ],
    "enrollment_identity_labels":
        enrollment_identity_labels,

    "exact_query_paths": [
        str(p) for p in exact_query_paths
    ],
    "exact_query_labels":
        exact_query_labels,

    "noisy_query_paths": [
        str(p) for p in noisy_query_paths
    ],
    "noisy_query_labels":
        noisy_query_labels,

    "unenrolled_query_paths": [
        str(p) for p in unenrolled_query_paths
    ],
    "unenrolled_query_labels":
        unenrolled_query_labels,

    "query_image_paths": [
        str(p) for p in query_image_paths
    ],
    "query_identity_labels":
        query_identity_labels,

    "query_group_labels":
        query_group_labels,

    "y_true_images":
        y_true_images
}

with open(
    PROTOCOL_FILE,
    "wb"
) as f:
    pickle.dump(
        protocol,
        f
    )

print("=" * 80)
print("PROTOCOL SAVED")
print("=" * 80)

print(PROTOCOL_FILE)

print(
    f"\nEnrollment : "
    f"{len(enrollment_image_paths):,}"
)

print(
    f"Queries    : "
    f"{len(query_image_paths):,}"
)

print(
    "\n✓ Safe to restart the notebook session."
)

In [1]:
# ============================================================
# CELL 26 — Verify ONNX Runtime GPU
# ============================================================

import onnxruntime as ort
import torch

print("=" * 80)
print("GPU RUNTIME VERIFICATION")
print("=" * 80)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "No CUDA GPU"
)

print(
    "\nONNX Runtime version:",
    ort.__version__
)

print(
    "\nAvailable providers:"
)

for provider in (
    ort.get_available_providers()
):
    print(
        " ",
        provider
    )


if (
    "CUDAExecutionProvider"
    in ort.get_available_providers()
):

    print(
        "\n✓ CUDAExecutionProvider is available."
    )

else:

    print(
        "\n✗ CUDAExecutionProvider is still missing."
    )

GPU RUNTIME VERIFICATION
GPU: Tesla T4

ONNX Runtime version: 1.30.0

Available providers:
  TensorrtExecutionProvider
  CUDAExecutionProvider
  CPUExecutionProvider

✓ CUDAExecutionProvider is available.


In [2]:
# ============================================================
# CELL 27 — Restore experiment protocol if necessary
# ============================================================

import pickle
import numpy as np
from pathlib import Path

PROTOCOL_FILE = Path(
    "/kaggle/working/ms1mv2_hbf_protocol.pkl"
)


if "enrollment_image_paths" not in globals():

    if not PROTOCOL_FILE.exists():

        raise FileNotFoundError(
            "Protocol file was not found. "
            "Rerun the dataset construction cells."
        )

    with open(
        PROTOCOL_FILE,
        "rb"
    ) as f:

        protocol = pickle.load(f)


    enrollment_image_paths = [
        Path(p)
        for p in protocol[
            "enrollment_image_paths"
        ]
    ]

    enrollment_identity_labels = protocol[
        "enrollment_identity_labels"
    ]


    exact_query_paths = [
        Path(p)
        for p in protocol[
            "exact_query_paths"
        ]
    ]

    exact_query_labels = protocol[
        "exact_query_labels"
    ]


    noisy_query_paths = [
        Path(p)
        for p in protocol[
            "noisy_query_paths"
        ]
    ]

    noisy_query_labels = protocol[
        "noisy_query_labels"
    ]


    unenrolled_query_paths = [
        Path(p)
        for p in protocol[
            "unenrolled_query_paths"
        ]
    ]

    unenrolled_query_labels = protocol[
        "unenrolled_query_labels"
    ]


    query_image_paths = [
        Path(p)
        for p in protocol[
            "query_image_paths"
        ]
    ]

    query_identity_labels = protocol[
        "query_identity_labels"
    ]

    query_group_labels = protocol[
        "query_group_labels"
    ]

    y_true_images = protocol[
        "y_true_images"
    ]


    print("✓ Protocol restored.")

else:

    print(
        "✓ Protocol variables already exist."
    )


print(
    f"Enrollment: "
    f"{len(enrollment_image_paths):,}"
)

print(
    f"Queries   : "
    f"{len(query_image_paths):,}"
)

✓ Protocol restored.
Enrollment: 30,000
Queries   : 906


In [3]:
# ============================================================
# CELL 28 — Initialize InsightFace model pack
# ============================================================

import insightface
from insightface.app import FaceAnalysis

print("=" * 80)
print("INITIALIZING INSIGHTFACE")
print("=" * 80)


app = FaceAnalysis(
    name="buffalo_l",
    providers=[
        "CUDAExecutionProvider",
        "CPUExecutionProvider"
    ]
)

app.prepare(
    ctx_id=0,
    det_size=(640, 640)
)


print("\n✓ InsightFace model pack initialized.")

INITIALIZING INSIGHTFACE
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 94332.09KB/s]


Failed to load libcublasLt.so.13: libcublasLt.so.13: cannot open shared object file: No such file or directory
Failed to load libcublas.so.13: libcublas.so.13: cannot open shared object file: No such file or directory
Failed to load libnvrtc.so.13: libnvrtc.so.13: cannot open shared object file: No such file or directory
Failed to load libcufft.so.12: libcufft.so.12: cannot open shared object file: No such file or directory
Failed to load libcudart.so.13: libcudart.so.13: cannot open shared object file: No such file or directory
Please follow https://onnxruntime.ai/docs/install/#cuda-and-cudnn to install CUDA.


2026-09-30 20:39:02.230126159 [E:onnxruntime:Default, provider_bridge_ort.cc:2395 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1988 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.12/dist-packages/onnxruntime/capi/libonnxruntime_providers_cuda.so with error: libcublasLt.so.13: cannot open shared object file: No such file or directory

2026-09-30 20:39:02.890325870 [E:onnxruntime:Default, provider_bridge_ort.cc:2395 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1988 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.12/dist-packages/onnxruntime/capi/libonnxruntime_providers_cuda.so with error: libcublasLt.so.13: cannot open shared object file: No such file or directory

2026-09-30 20:39:02.921975070 [E:onnxruntime:Default, prov


✓ InsightFace model pack initialized.


In [4]:
# ============================================================
# CELL 29 — Extract ArcFace recognition model
# ============================================================

recognition_model = None


print("=" * 80)
print("INSIGHTFACE MODELS")
print("=" * 80)


for model in app.models.values():

    print(
        type(model).__name__,
        "| task:",
        getattr(model, "taskname", "unknown")
    )

    if (
        getattr(
            model,
            "taskname",
            None
        )
        == "recognition"
    ):

        recognition_model = model


if recognition_model is None:

    raise RuntimeError(
        "Could not locate InsightFace "
        "recognition model."
    )


print(
    "\nRecognition model:",
    type(recognition_model).__name__
)

print(
    "Input size:",
    recognition_model.input_size
)

print(
    "\n✓ ArcFace recognition model ready."
)

INSIGHTFACE MODELS
Landmark | task: landmark_3d_68
Landmark | task: landmark_2d_106
SCRFD | task: detection
Attribute | task: genderage
ArcFaceONNX | task: recognition

Recognition model: ArcFaceONNX
Input size: (112, 112)

✓ ArcFace recognition model ready.


In [5]:
# ============================================================
# CELL 30 — Test direct ArcFace embedding
# ============================================================

import cv2
import numpy as np


test_path = enrollment_image_paths[0]


# ------------------------------------------------------------
# Read aligned MS1MV2 image
# ------------------------------------------------------------

img = cv2.imread(
    str(test_path)
)

if img is None:

    raise RuntimeError(
        f"Could not read:\n{test_path}"
    )


# ------------------------------------------------------------
# MS1MV2 should already be 112x112.
# Resize only as a safety measure.
# ------------------------------------------------------------

if img.shape[:2] != (112, 112):

    img = cv2.resize(
        img,
        (112, 112)
    )


# ------------------------------------------------------------
# Direct ArcFace feature extraction
# ------------------------------------------------------------

embedding = recognition_model.get_feat(
    img
)


embedding = np.asarray(
    embedding,
    dtype=np.float32
).reshape(-1)


# ------------------------------------------------------------
# Explicit L2 normalization
# ------------------------------------------------------------

embedding = (
    embedding
    /
    (
        np.linalg.norm(
            embedding
        )
        + 1e-12
    )
)


print("=" * 80)
print("ARCFACE SINGLE-IMAGE TEST")
print("=" * 80)

print(
    "Image:",
    test_path
)

print(
    "Image shape:",
    img.shape
)

print(
    "Embedding shape:",
    embedding.shape
)

print(
    "Embedding norm:",
    np.linalg.norm(
        embedding
    )
)

print(
    "\nFirst 10 dimensions:"
)

print(
    embedding[:10]
)


assert embedding.shape == (
    512,
)

assert np.isfinite(
    embedding
).all()


print(
    "\n✓ Direct 512-D ArcFace embedding successful."
)

ARCFACE SINGLE-IMAGE TEST
Image: /kaggle/input/datasets/yakhyokhuja/ms1m-arcface-dataset/ms1m-arcface/15481/1223566.jpg
Image shape: (112, 112, 3)
Embedding shape: (512,)
Embedding norm: 1.0

First 10 dimensions:
[-0.00932127 -0.0283124  -0.0114073  -0.03890664  0.02975851 -0.04028832
  0.02057926  0.0129394  -0.0437858   0.00179577]

✓ Direct 512-D ArcFace embedding successful.


In [6]:
# ============================================================
# CELL 31 — ArcFace same/different identity sanity check
# ============================================================

from pathlib import Path
import cv2
import numpy as np


def arcface_embedding(image_path):

    img = cv2.imread(
        str(image_path)
    )

    if img is None:
        raise RuntimeError(
            f"Could not read {image_path}"
        )

    if img.shape[:2] != (112, 112):

        img = cv2.resize(
            img,
            (112, 112)
        )

    feat = recognition_model.get_feat(
        img
    )

    feat = np.asarray(
        feat,
        dtype=np.float32
    ).reshape(-1)

    feat /= (
        np.linalg.norm(feat)
        + 1e-12
    )

    return feat


# ------------------------------------------------------------
# Identity A
# ------------------------------------------------------------

anchor_path = Path(
    enrollment_image_paths[0]
)

anchor_identity = (
    enrollment_identity_labels[0]
)

anchor_folder = (
    anchor_path.parent
)


same_candidates = [
    p
    for p in anchor_folder.iterdir()
    if (
        p.is_file()
        and
        p != anchor_path
        and
        p.suffix.lower()
        in {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp"
        }
    )
]


if not same_candidates:

    raise RuntimeError(
        "First identity does not contain "
        "a second usable image."
    )


same_path = same_candidates[0]


# ------------------------------------------------------------
# Different identity B
# ------------------------------------------------------------

different_path = Path(
    enrollment_image_paths[1]
)

different_identity = (
    enrollment_identity_labels[1]
)


# ------------------------------------------------------------
# Extract embeddings
# ------------------------------------------------------------

anchor_emb = arcface_embedding(
    anchor_path
)

same_emb = arcface_embedding(
    same_path
)

different_emb = arcface_embedding(
    different_path
)


# Because embeddings are L2 normalized,
# dot product = cosine similarity.

same_similarity = float(
    np.dot(
        anchor_emb,
        same_emb
    )
)

different_similarity = float(
    np.dot(
        anchor_emb,
        different_emb
    )
)


print("=" * 80)
print("ARCFACE SANITY CHECK")
print("=" * 80)

print(
    "Anchor identity    :",
    anchor_identity
)

print(
    "Same identity      :",
    anchor_identity
)

print(
    "Different identity :",
    different_identity
)


print("\nCosine similarities:")

print(
    f"  Same person      : "
    f"{same_similarity:.4f}"
)

print(
    f"  Different person : "
    f"{different_similarity:.4f}"
)

print(
    f"  Difference       : "
    f"{same_similarity - different_similarity:.4f}"
)


if (
    same_similarity
    >
    different_similarity
):

    print(
        "\n✓ Sanity check passed."
    )

else:

    print(
        "\n⚠ Same-person similarity was not "
        "higher for this particular pair."
    )

ARCFACE SANITY CHECK
Anchor identity    : 15481
Same identity      : 15481
Different identity : 57269

Cosine similarities:
  Same person      : 0.5462
  Different person : 0.0876
  Difference       : 0.4587

✓ Sanity check passed.


In [7]:
# ============================================================
# CELL 32 — Batch ArcFace embedding extraction
# ============================================================

import cv2
import numpy as np
import time
from pathlib import Path

BATCH_SIZE = 256

EMBEDDING_CACHE = Path(
    "/kaggle/working/ms1mv2_arcface_embeddings.npz"
)


# ------------------------------------------------------------
# Unique physical paths requiring embeddings
#
# Exact queries are already enrollment images.
# Noisy queries are new PNGs.
# Unenrolled queries are new MS1MV2 images.
# ------------------------------------------------------------

all_required_paths = (
    list(enrollment_image_paths)
    +
    list(noisy_query_paths)
    +
    list(unenrolled_query_paths)
)

unique_paths = list(
    dict.fromkeys(
        str(Path(p))
        for p in all_required_paths
    )
)


print("=" * 80)
print("ARCFACE EMBEDDING EXTRACTION")
print("=" * 80)

print(
    f"Enrollment images : "
    f"{len(enrollment_image_paths):,}"
)

print(
    f"Noisy queries     : "
    f"{len(noisy_query_paths):,}"
)

print(
    f"Unenrolled       : "
    f"{len(unenrolled_query_paths):,}"
)

print(
    f"Unique files to encode: "
    f"{len(unique_paths):,}"
)

print(
    f"Batch size       : "
    f"{BATCH_SIZE}"
)


# ------------------------------------------------------------
# Image loader
# ------------------------------------------------------------

def load_arcface_image(path):

    image = cv2.imread(
        str(path)
    )

    if image is None:

        raise RuntimeError(
            f"Could not read image:\n{path}"
        )

    if image.shape[:2] != (
        112,
        112
    ):

        image = cv2.resize(
            image,
            (112, 112),
            interpolation=cv2.INTER_LINEAR
        )

    return image


# ------------------------------------------------------------
# Extract embeddings
# ------------------------------------------------------------

all_embeddings = np.empty(
    (
        len(unique_paths),
        512
    ),
    dtype=np.float32
)


start_time = time.time()


for start_idx in range(
    0,
    len(unique_paths),
    BATCH_SIZE
):

    end_idx = min(
        start_idx + BATCH_SIZE,
        len(unique_paths)
    )

    batch_paths = unique_paths[
        start_idx:end_idx
    ]


    # --------------------------------------------------------
    # Load batch
    # --------------------------------------------------------

    batch_images = [
        load_arcface_image(path)
        for path in batch_paths
    ]


    # --------------------------------------------------------
    # ArcFace batch inference
    # --------------------------------------------------------

    features = (
        recognition_model.get_feat(
            batch_images
        )
    )

    features = np.asarray(
        features,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # L2 normalization
    # --------------------------------------------------------

    norms = np.linalg.norm(
        features,
        axis=1,
        keepdims=True
    )

    features = (
        features
        /
        np.maximum(
            norms,
            1e-12
        )
    )


    all_embeddings[
        start_idx:end_idx
    ] = features


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    processed = end_idx

    elapsed = (
        time.time()
        - start_time
    )

    speed = (
        processed
        /
        elapsed
    )

    remaining = (
        len(unique_paths)
        -
        processed
    )

    eta = (
        remaining
        /
        speed
        if speed > 0
        else 0
    )


    print(
        f"\r"
        f"Processed "
        f"{processed:6d}/"
        f"{len(unique_paths):6d} | "
        f"{speed:7.2f} img/s | "
        f"ETA {eta/60:6.2f} min",
        end="",
        flush=True
    )


print()


total_time = (
    time.time()
    -
    start_time
)


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert all_embeddings.shape == (
    len(unique_paths),
    512
)

assert np.isfinite(
    all_embeddings
).all()


norms = np.linalg.norm(
    all_embeddings,
    axis=1
)


print("\n" + "=" * 80)
print("EXTRACTION COMPLETE")
print("=" * 80)

print(
    f"Embeddings : "
    f"{all_embeddings.shape}"
)

print(
    f"Runtime    : "
    f"{total_time/60:.2f} minutes"
)

print(
    f"Throughput : "
    f"{len(unique_paths)/total_time:.2f} images/sec"
)

print(
    f"Mean norm  : "
    f"{norms.mean():.6f}"
)

print(
    f"Min norm   : "
    f"{norms.min():.6f}"
)

print(
    f"Max norm   : "
    f"{norms.max():.6f}"
)

print(
    "\n✓ All ArcFace embeddings generated."
)

ARCFACE EMBEDDING EXTRACTION
Enrollment images : 30,000
Noisy queries     : 302
Unenrolled       : 302
Unique files to encode: 30,604
Batch size       : 256
Processed  30604/ 30604 |    4.31 img/s | ETA   0.00 min

EXTRACTION COMPLETE
Embeddings : (30604, 512)
Runtime    : 118.46 minutes
Throughput : 4.31 images/sec
Mean norm  : 1.000000
Min norm   : 1.000000
Max norm   : 1.000000

✓ All ArcFace embeddings generated.


In [8]:
# ============================================================
# CELL 33 — Save ArcFace embedding cache
# ============================================================

np.savez_compressed(
    EMBEDDING_CACHE,
    paths=np.asarray(
        unique_paths
    ),
    embeddings=all_embeddings
)


path_to_index = {
    path: i
    for i, path
    in enumerate(unique_paths)
}


def get_cached_embedding(path):

    key = str(
        Path(path)
    )

    return all_embeddings[
        path_to_index[key]
    ]


print("=" * 80)
print("EMBEDDING CACHE")
print("=" * 80)

print(
    "Saved:",
    EMBEDDING_CACHE
)

print(
    f"Entries: "
    f"{len(path_to_index):,}"
)

print(
    f"Dimensions: "
    f"{all_embeddings.shape[1]}"
)


# Test exact-query lookup
test_enrollment = get_cached_embedding(
    exact_query_paths[0]
)

print(
    f"\nTest embedding shape: "
    f"{test_enrollment.shape}"
)

print(
    f"Test norm: "
    f"{np.linalg.norm(test_enrollment):.6f}"
)

print(
    "\n✓ Cache ready."
)

EMBEDDING CACHE
Saved: /kaggle/working/ms1mv2_arcface_embeddings.npz
Entries: 30,604
Dimensions: 512

Test embedding shape: (512,)
Test norm: 1.000000

✓ Cache ready.


In [9]:
# ============================================================
# CELL 34 — ArcFace query diagnostics
# ============================================================

# ------------------------------------------------------------
# Enrollment matrix
# ------------------------------------------------------------

enrollment_embeddings = np.stack(
    [
        get_cached_embedding(p)
        for p in enrollment_image_paths
    ]
).astype(
    np.float32
)


# ------------------------------------------------------------
# Exact query matrix
# ------------------------------------------------------------

exact_embeddings = np.stack(
    [
        get_cached_embedding(p)
        for p in exact_query_paths
    ]
).astype(
    np.float32
)


# ------------------------------------------------------------
# Noisy query matrix
# ------------------------------------------------------------

noisy_embeddings = np.stack(
    [
        get_cached_embedding(p)
        for p in noisy_query_paths
    ]
).astype(
    np.float32
)


# ------------------------------------------------------------
# Unenrolled matrix
# ------------------------------------------------------------

unenrolled_embeddings = np.stack(
    [
        get_cached_embedding(p)
        for p in unenrolled_query_paths
    ]
).astype(
    np.float32
)


print("=" * 80)
print("EMBEDDING MATRICES")
print("=" * 80)

print(
    "Enrollment:",
    enrollment_embeddings.shape
)

print(
    "Exact     :",
    exact_embeddings.shape
)

print(
    "Noisy     :",
    noisy_embeddings.shape
)

print(
    "Unenrolled:",
    unenrolled_embeddings.shape
)


# ------------------------------------------------------------
# Exact vs noisy paired cosine similarity
# ------------------------------------------------------------

exact_noisy_similarity = np.sum(
    exact_embeddings
    *
    noisy_embeddings,
    axis=1
)


print("\n" + "=" * 80)
print("EXACT ↔ 1% NOISY")
print("=" * 80)

print(
    f"Mean   : "
    f"{exact_noisy_similarity.mean():.4f}"
)

print(
    f"Median : "
    f"{np.median(exact_noisy_similarity):.4f}"
)

print(
    f"Min    : "
    f"{exact_noisy_similarity.min():.4f}"
)

print(
    f"Max    : "
    f"{exact_noisy_similarity.max():.4f}"
)

EMBEDDING MATRICES
Enrollment: (30000, 512)
Exact     : (302, 512)
Noisy     : (302, 512)
Unenrolled: (302, 512)

EXACT ↔ 1% NOISY
Mean   : 0.9753
Median : 0.9778
Min    : 0.9212
Max    : 0.9906


In [10]:
# ============================================================
# CELL 35 — Unenrolled 1:N ArcFace similarity
# ============================================================

import time

print("=" * 80)
print("UNENROLLED → 30,000 WATCHLIST")
print("=" * 80)


start = time.time()


# 302 x 512 @ 512 x 30000
#
# Result:
# 302 x 30000 cosine similarities

unenrolled_scores = (
    unenrolled_embeddings
    @
    enrollment_embeddings.T
)


max_unenrolled_similarity = (
    unenrolled_scores.max(
        axis=1
    )
)


best_match_indices = (
    unenrolled_scores.argmax(
        axis=1
    )
)


elapsed = (
    time.time()
    -
    start
)


print(
    f"Comparisons: "
    f"{302 * 30000:,}"
)

print(
    f"Runtime: "
    f"{elapsed:.3f} sec"
)


print(
    "\nMaximum watchlist similarity "
    "for each unseen query:"
)

print(
    f"Mean   : "
    f"{max_unenrolled_similarity.mean():.4f}"
)

print(
    f"Median : "
    f"{np.median(max_unenrolled_similarity):.4f}"
)

print(
    f"Min    : "
    f"{max_unenrolled_similarity.min():.4f}"
)

print(
    f"Max    : "
    f"{max_unenrolled_similarity.max():.4f}"
)


print(
    "\nPercentiles:"
)

for percentile in [
    50,
    90,
    95,
    99
]:

    value = np.percentile(
        max_unenrolled_similarity,
        percentile
    )

    print(
        f"  P{percentile:<2}: "
        f"{value:.4f}"
    )


print(
    "\n✓ 1:N ArcFace diagnostic complete."
)

UNENROLLED → 30,000 WATCHLIST
Comparisons: 9,060,000
Runtime: 0.072 sec

Maximum watchlist similarity for each unseen query:
Mean   : 0.2926
Median : 0.2798
Min    : 0.2095
Max    : 0.6744

Percentiles:
  P50: 0.2798
  P90: 0.3343
  P95: 0.3484
  P99: 0.5939

✓ 1:N ArcFace diagnostic complete.


In [11]:
# ============================================================
# CELL 36 — Fixed ArcFace -> 4900-bit projection
# ============================================================

import numpy as np
from pathlib import Path

PROJECTION_SEED = 2026

EMBED_DIM = 512
BINARY_DIM = 4900

projection_rng = np.random.default_rng(
    PROJECTION_SEED
)


# ------------------------------------------------------------
# Random hyperplanes
#
# Each column represents one random hyperplane.
# ------------------------------------------------------------

projection_matrix = (
    projection_rng.standard_normal(
        (
            EMBED_DIM,
            BINARY_DIM
        )
    )
    .astype(
        np.float32
    )
)


# ------------------------------------------------------------
# Normalize each hyperplane
# ------------------------------------------------------------

projection_matrix /= (
    np.linalg.norm(
        projection_matrix,
        axis=0,
        keepdims=True
    )
    + 1e-12
)


print("=" * 80)
print("RANDOM-HYPERPLANE PROJECTION")
print("=" * 80)

print(
    "Projection seed :",
    PROJECTION_SEED
)

print(
    "Input dimension :",
    EMBED_DIM
)

print(
    "Output bits     :",
    BINARY_DIM
)

print(
    "Matrix shape    :",
    projection_matrix.shape
)

print(
    "Matrix size     : "
    f"{projection_matrix.nbytes / 1024**2:.2f} MB"
)


# ------------------------------------------------------------
# Save it
# ------------------------------------------------------------

PROJECTION_FILE = Path(
    "/kaggle/working/"
    "arcface_512_to_4900_projection.npy"
)

np.save(
    PROJECTION_FILE,
    projection_matrix
)

print(
    "\nSaved:",
    PROJECTION_FILE
)

print(
    "\n✓ Fixed projection created."
)

RANDOM-HYPERPLANE PROJECTION
Projection seed : 2026
Input dimension : 512
Output bits     : 4900
Matrix shape    : (512, 4900)
Matrix size     : 9.57 MB

Saved: /kaggle/working/arcface_512_to_4900_projection.npy

✓ Fixed projection created.


In [12]:
# ============================================================
# CELL 37 — Convert ArcFace embeddings -> 4900-bit templates
# ============================================================

PROJECTION_BATCH = 1024


def embeddings_to_binary(
    embeddings,
    projection,
    batch_size=1024
):

    n = embeddings.shape[0]

    output = np.empty(
        (
            n,
            projection.shape[1]
        ),
        dtype=np.uint8
    )

    for start in range(
        0,
        n,
        batch_size
    ):

        end = min(
            start + batch_size,
            n
        )

        projected = (
            embeddings[start:end]
            @
            projection
        )

        output[start:end] = (
            projected >= 0
        ).astype(
            np.uint8
        )

    return output


print("=" * 80)
print("GENERATING 4900-BIT BIOMETRIC TEMPLATES")
print("=" * 80)


enrollment_binary = embeddings_to_binary(
    enrollment_embeddings,
    projection_matrix,
    PROJECTION_BATCH
)

exact_binary = embeddings_to_binary(
    exact_embeddings,
    projection_matrix,
    PROJECTION_BATCH
)

noisy_binary = embeddings_to_binary(
    noisy_embeddings,
    projection_matrix,
    PROJECTION_BATCH
)

unenrolled_binary = embeddings_to_binary(
    unenrolled_embeddings,
    projection_matrix,
    PROJECTION_BATCH
)


print(
    "Enrollment:",
    enrollment_binary.shape
)

print(
    "Exact     :",
    exact_binary.shape
)

print(
    "Noisy     :",
    noisy_binary.shape
)

print(
    "Unenrolled:",
    unenrolled_binary.shape
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert enrollment_binary.shape == (
    30000,
    4900
)

assert exact_binary.shape == (
    302,
    4900
)

assert noisy_binary.shape == (
    302,
    4900
)

assert unenrolled_binary.shape == (
    302,
    4900
)

assert np.all(
    np.isin(
        enrollment_binary,
        [0, 1]
    )
)


print(
    "\n✓ All templates are binary and 4,900 bits long."
)

GENERATING 4900-BIT BIOMETRIC TEMPLATES
Enrollment: (30000, 4900)
Exact     : (302, 4900)
Noisy     : (302, 4900)
Unenrolled: (302, 4900)

✓ All templates are binary and 4,900 bits long.


In [13]:
# ============================================================
# CELL 38 — Verify exact-query binary templates
# ============================================================

# Map enrollment identity -> row
enrollment_id_to_index = {
    identity: i
    for i, identity
    in enumerate(
        enrollment_identity_labels
    )
}


exact_reference_binary = np.stack(
    [
        enrollment_binary[
            enrollment_id_to_index[
                identity
            ]
        ]
        for identity
        in exact_query_labels
    ]
)


exact_hd_bits = np.sum(
    exact_binary
    !=
    exact_reference_binary,
    axis=1
)

exact_hd_fraction = (
    exact_hd_bits
    /
    BINARY_DIM
)


print("=" * 80)
print("EXACT ENROLLED — HAMMING DISTANCE")
print("=" * 80)

print(
    f"Mean bits different : "
    f"{exact_hd_bits.mean():.2f}"
)

print(
    f"Min                : "
    f"{exact_hd_bits.min()}"
)

print(
    f"Max                : "
    f"{exact_hd_bits.max()}"
)

print(
    f"Mean HD fraction   : "
    f"{exact_hd_fraction.mean():.6f}"
)


assert np.all(
    exact_hd_bits == 0
)


print(
    "\n✓ Exact queries reproduce their enrollment templates."
)

EXACT ENROLLED — HAMMING DISTANCE
Mean bits different : 0.00
Min                : 0
Max                : 0
Mean HD fraction   : 0.000000

✓ Exact queries reproduce their enrollment templates.


In [14]:
# ============================================================
# CELL 39 — 1%-noise Hamming-distance analysis
# ============================================================

noisy_hd_bits = np.sum(
    noisy_binary
    !=
    exact_reference_binary,
    axis=1
)

noisy_hd_fraction = (
    noisy_hd_bits
    /
    BINARY_DIM
)


print("=" * 80)
print("1% IMAGE NOISE → BINARY TEMPLATE CHANGE")
print("=" * 80)

print(
    f"Mean changed bits : "
    f"{noisy_hd_bits.mean():.2f} / 4900"
)

print(
    f"Median            : "
    f"{np.median(noisy_hd_bits):.2f}"
)

print(
    f"Minimum           : "
    f"{noisy_hd_bits.min()}"
)

print(
    f"Maximum           : "
    f"{noisy_hd_bits.max()}"
)

print(
    f"\nMean HD           : "
    f"{noisy_hd_fraction.mean():.4f}"
)

print(
    f"Median HD         : "
    f"{np.median(noisy_hd_fraction):.4f}"
)

print(
    f"Min HD            : "
    f"{noisy_hd_fraction.min():.4f}"
)

print(
    f"Max HD            : "
    f"{noisy_hd_fraction.max():.4f}"
)

1% IMAGE NOISE → BINARY TEMPLATE CHANGE
Mean changed bits : 340.18 / 4900
Median            : 331.50
Minimum           : 198
Maximum           : 668

Mean HD           : 0.0694
Median HD         : 0.0677
Min HD            : 0.0404
Max HD            : 0.1363


In [15]:
# ============================================================
# CELL 40 — Unenrolled nearest-template Hamming analysis
# ============================================================

QUERY_BATCH = 16

min_unenrolled_hd_bits = np.empty(
    302,
    dtype=np.int32
)

nearest_enrollment_index = np.empty(
    302,
    dtype=np.int32
)


print("=" * 80)
print("UNENROLLED → 30,000 BINARY WATCHLIST")
print("=" * 80)


for start in range(
    0,
    len(unenrolled_binary),
    QUERY_BATCH
):

    end = min(
        start + QUERY_BATCH,
        len(unenrolled_binary)
    )

    q = unenrolled_binary[
        start:end
    ]

    # Shape:
    # B x 30000 x 4900
    #
    # Keep batch deliberately small.
    distances = np.count_nonzero(
        q[:, None, :]
        !=
        enrollment_binary[None, :, :],
        axis=2
    )

    min_unenrolled_hd_bits[
        start:end
    ] = distances.min(
        axis=1
    )

    nearest_enrollment_index[
        start:end
    ] = distances.argmin(
        axis=1
    )

    print(
        f"\rProcessed "
        f"{end:3d}/302",
        end="",
        flush=True
    )


print()


min_unenrolled_hd_fraction = (
    min_unenrolled_hd_bits
    /
    BINARY_DIM
)


print(
    "\nNearest enrollment-template HD:"
)

print(
    f"Mean bits   : "
    f"{min_unenrolled_hd_bits.mean():.2f}"
)

print(
    f"Median bits : "
    f"{np.median(min_unenrolled_hd_bits):.2f}"
)

print(
    f"Min bits    : "
    f"{min_unenrolled_hd_bits.min()}"
)

print(
    f"Max bits    : "
    f"{min_unenrolled_hd_bits.max()}"
)


print(
    "\nNormalized HD:"
)

print(
    f"Mean   : "
    f"{min_unenrolled_hd_fraction.mean():.4f}"
)

print(
    f"Median : "
    f"{np.median(min_unenrolled_hd_fraction):.4f}"
)

print(
    f"Min    : "
    f"{min_unenrolled_hd_fraction.min():.4f}"
)

print(
    f"Max    : "
    f"{min_unenrolled_hd_fraction.max():.4f}"
)


print("\nPercentiles:")

for p in [
    1,
    5,
    10,
    50,
    90,
    95,
    99
]:

    print(
        f"  P{p:<2}: "
        f"{np.percentile(min_unenrolled_hd_fraction, p):.4f}"
    )


print(
    "\n✓ Binary-space diagnostic complete."
)

UNENROLLED → 30,000 BINARY WATCHLIST
Processed 302/302

Nearest enrollment-template HD:
Mean bits   : 1969.08
Median bits : 1987.00
Min bits    : 1290
Max bits    : 2102

Normalized HD:
Mean   : 0.4019
Median : 0.4055
Min    : 0.2633
Max    : 0.4290

Percentiles:
  P1 : 0.2930
  P5 : 0.3827
  P10: 0.3894
  P50: 0.4055
  P90: 0.4157
  P95: 0.4186
  P99: 0.4218

✓ Binary-space diagnostic complete.


In [16]:
# ============================================================
# CELL 41 — Convert binary templates to HBF string format
# ============================================================

def binary_matrix_to_strings(matrix):

    return [
        "".join(
            row.astype(str)
        )
        for row in matrix
    ]


print("=" * 80)
print("CONVERTING TEMPLATES TO ORIGINAL HBF FORMAT")
print("=" * 80)


# ------------------------------------------------------------
# Enrollment
# ------------------------------------------------------------

enrollment_data_ms1 = (
    binary_matrix_to_strings(
        enrollment_binary
    )
)


# ------------------------------------------------------------
# Query groups
# ------------------------------------------------------------

exact_query_data_ms1 = (
    binary_matrix_to_strings(
        exact_binary
    )
)

noisy_query_data_ms1 = (
    binary_matrix_to_strings(
        noisy_binary
    )
)

unenrolled_query_data_ms1 = (
    binary_matrix_to_strings(
        unenrolled_binary
    )
)


# ------------------------------------------------------------
# Final 906-query sequence
#
# 0:302   Exact
# 302:604 Noisy
# 604:906 Unenrolled
# ------------------------------------------------------------

query_data_ms1 = (
    exact_query_data_ms1
    +
    noisy_query_data_ms1
    +
    unenrolled_query_data_ms1
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(
    enrollment_data_ms1
) == 30000

assert len(
    query_data_ms1
) == 906


assert all(
    len(x) == 4900
    for x in enrollment_data_ms1
)

assert all(
    len(x) == 4900
    for x in query_data_ms1
)


assert all(
    set(x).issubset(
        {"0", "1"}
    )
    for x in enrollment_data_ms1
)

assert all(
    set(x).issubset(
        {"0", "1"}
    )
    for x in query_data_ms1
)


print(
    f"Enrollment templates : "
    f"{len(enrollment_data_ms1):,}"
)

print(
    f"Query templates      : "
    f"{len(query_data_ms1):,}"
)

print(
    f"Template length      : "
    f"{len(enrollment_data_ms1[0]):,} bits"
)

print("\nQuery composition:")
print("  [0:302]   Exact enrolled")
print("  [302:604] 1% noisy enrolled")
print("  [604:906] Unenrolled")

print(
    "\n✓ HBF-formatted templates ready."
)

CONVERTING TEMPLATES TO ORIGINAL HBF FORMAT
Enrollment templates : 30,000
Query templates      : 906
Template length      : 4,900 bits

Query composition:
  [0:302]   Exact enrolled
  [302:604] 1% noisy enrolled
  [604:906] Unenrolled

✓ HBF-formatted templates ready.


In [17]:
# ============================================================
# CELL 42 — Original HBF configuration
# ============================================================

import math

n1 = 30000

m = 140530
k = 4

total_level = 3

d1 = [
    0,
    1,
    2
]

d1_l = [
    32,
    416,
    4900
]

d1_BFs = [
    153,
    11,
    1
]

TOTAL_BFS = sum(
    d1_BFs
)

THRESHOLD = 21


print("=" * 80)
print("ORIGINAL HBF CONFIGURATION")
print("=" * 80)

print(
    f"Enrollment templates : "
    f"{n1:,}"
)

print(
    f"Bloom-filter size    : "
    f"{m:,}"
)

print(
    f"Hash functions       : "
    f"{k}"
)

print(
    f"Levels               : "
    f"{total_level}"
)

print(
    f"Segment lengths      : "
    f"{d1_l}"
)

print(
    f"BFs per level        : "
    f"{d1_BFs}"
)

print(
    f"Total Bloom filters  : "
    f"{TOTAL_BFS}"
)

print(
    f"Original threshold   : "
    f">{THRESHOLD}"
)

print(
    "\n✓ Original HBF configuration restored."
)

ORIGINAL HBF CONFIGURATION
Enrollment templates : 30,000
Bloom-filter size    : 140,530
Hash functions       : 4
Levels               : 3
Segment lengths      : [32, 416, 4900]
BFs per level        : [153, 11, 1]
Total Bloom filters  : 165
Original threshold   : >21

✓ Original HBF configuration restored.


In [21]:
# ============================================================
# CELL 43A-1 — Install fnv
# ============================================================

import sys
import subprocess

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "fnv"
    ]
)

print("✓ fnv installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 1.4 MB/s eta 0:00:00
✓ fnv installed.


In [22]:
import math
import hashlib
import fnv

print("✓ HBF dependencies imported.")

✓ HBF dependencies imported.


In [23]:
# ============================================================
# CELL 43B — Original HBF hash function
# ============================================================

def get_hash_indices(input_segment, m, k):

    bits_to_take = math.floor(
        math.log2(m)
    )

    mask = (
        (1 << bits_to_take)
        - 1
    )

    hash_indices = []

    for key in range(
        1,
        k + 1
    ):

        key_bin = format(
            key,
            "08b"
        )

        combined = (
            input_segment.encode("utf-8")
            +
            key_bin.encode("utf-8")
        )

        sha_digest = hashlib.sha256(
            combined
        ).hexdigest()

        fnv_value = fnv.hash(
            sha_digest.encode("utf-8"),
            algorithm=fnv.fnv_1a,
            bits=32
        )

        fnv_bin = format(
            fnv_value,
            "032b"
        )

        fnv_int = int(
            fnv_bin,
            2
        )

        folded = (
            fnv_int
            ^
            (
                fnv_int
                >>
                bits_to_take
            )
        )

        index = (
            folded
            &
            mask
        )

        hash_indices.append(
            index
        )

    return hash_indices


print("✓ Original hash helper defined.")

✓ Original hash helper defined.


In [24]:
# ============================================================
# CELL 43C — Original HBF enrollment
# ============================================================

def bloom_filter_enrolling(
    enrollment_data,
    d1,
    d1_l,
    d1_BFs,
    m,
    k
):

    all_levels = []

    for level in range(
        len(d1)
    ):

        len_block = d1_l[level]

        num_of_BFs_in_that_layer = (
            d1_BFs[level]
        )

        level_filters = [
            bytearray(m)
            for _ in range(
                num_of_BFs_in_that_layer
            )
        ]


        for template in enrollment_data:

            c1 = 0

            for j in range(
                num_of_BFs_in_that_layer
            ):

                # Original code behavior retained
                if (
                    j
                    ==
                    num_of_BFs_in_that_layer
                ):

                    temp_input = (
                        template[c1:]
                    )

                else:

                    temp_input = (
                        template[
                            c1:
                            c1
                            +
                            len_block
                        ]
                    )

                    c1 += len_block


                indices = get_hash_indices(
                    temp_input,
                    m,
                    k
                )

                bf = level_filters[j]

                for index in indices:
                    bf[index] = 1


        all_levels.append(
            level_filters
        )

    return all_levels


print("✓ Original HBF enrollment function defined.")

✓ Original HBF enrollment function defined.


In [25]:
# ============================================================
# CELL 43D — Original HBF query
# ============================================================

def bloom_filter_querying(
    query_template,
    BF_data,
    d1,
    d1_l,
    d1_BFs,
    m,
    k
):

    authenticated_per_level = []

    for level in range(
        len(d1)
    ):

        len_block = d1_l[level]

        num_of_BFs_in_that_layer = (
            d1_BFs[level]
        )

        authenticated_count = 0

        c1 = 0


        for j in range(
            num_of_BFs_in_that_layer
        ):

            # Original code behavior retained
            if (
                j
                ==
                num_of_BFs_in_that_layer
            ):

                temp_input = (
                    query_template[c1:]
                )

            else:

                temp_input = (
                    query_template[
                        c1:
                        c1
                        +
                        len_block
                    ]
                )

                c1 += len_block


            indices = get_hash_indices(
                temp_input,
                m,
                k
            )


            bf = BF_data[
                level
            ][j]


            authenticated = all(
                bf[index] == 1
                for index
                in indices
            )


            if authenticated:
                authenticated_count += 1


        authenticated_per_level.append(
            authenticated_count
        )

    return authenticated_per_level


print("✓ Original HBF query function defined.")

✓ Original HBF query function defined.


In [26]:
# ============================================================
# CELL 43E — Verify functions
# ============================================================

required_functions = [
    "get_hash_indices",
    "bloom_filter_enrolling",
    "bloom_filter_querying"
]

for name in required_functions:

    print(
        f"{name}:",
        "OK"
        if name in globals()
        else "MISSING"
    )

get_hash_indices: OK
bloom_filter_enrolling: OK
bloom_filter_querying: OK


In [27]:
# ============================================================
# CELL 44 — Enroll 30,000 MS1MV2 templates into HBF
# ============================================================

import time

print("=" * 80)
print("MS1MV2 → ORIGINAL HBF ENROLLMENT")
print("=" * 80)

print(
    f"Templates: "
    f"{len(enrollment_data_ms1):,}"
)

print(
    "Starting enrollment..."
)


start_time = time.time()


BF_data_ms1 = bloom_filter_enrolling(
    enrollment_data_ms1,
    d1,
    d1_l,
    d1_BFs,
    m,
    k
)


enrollment_time = (
    time.time()
    -
    start_time
)


print("\n" + "=" * 80)
print("ENROLLMENT COMPLETE")
print("=" * 80)

print(
    f"Total time : "
    f"{enrollment_time:.2f} sec"
)

print(
    f"Per image  : "
    f"{enrollment_time / 30000 * 1000:.4f} ms"
)

print(
    "\n✓ 30,000 MS1MV2 templates enrolled."
)

MS1MV2 → ORIGINAL HBF ENROLLMENT
Templates: 30,000
Starting enrollment...

ENROLLMENT COMPLETE
Total time : 578.80 sec
Per image  : 19.2934 ms

✓ 30,000 MS1MV2 templates enrolled.


In [28]:
# ============================================================
# CELL 45 — Query original HBF
# ============================================================

print("=" * 80)
print("MS1MV2 → ORIGINAL HBF QUERY")
print("=" * 80)

print(
    f"Queries: "
    f"{len(query_data_ms1)}"
)


query_results_ms1 = []

start_time = time.time()


for i, query_template in enumerate(
    query_data_ms1
):

    result = bloom_filter_querying(
        query_template,
        BF_data_ms1,
        d1,
        d1_l,
        d1_BFs,
        m,
        k
    )

    query_results_ms1.append(
        result
    )

    if (
        (i + 1) % 100 == 0
        or
        (i + 1) == 906
    ):

        print(
            f"\rQueried "
            f"{i + 1:3d}/906",
            end="",
            flush=True
        )


query_time = (
    time.time()
    -
    start_time
)

print()


print("\n" + "=" * 80)
print("QUERY COMPLETE")
print("=" * 80)

print(
    f"Total query time : "
    f"{query_time:.4f} sec"
)

print(
    f"Average/query    : "
    f"{query_time / 906 * 1000:.4f} ms"
)

MS1MV2 → ORIGINAL HBF QUERY
Queries: 906
Queried 906/906

QUERY COMPLETE
Total query time : 17.5435 sec
Average/query    : 19.3637 ms


In [29]:
# ============================================================
# CELL 46 — Aggregate HBF query scores
# ============================================================

import numpy as np

# Each query result should look like:
# [level0_count, level1_count, level2_count]

query_results_array = np.asarray(
    query_results_ms1,
    dtype=np.int32
)

assert query_results_array.shape == (
    906,
    3
)

authenticated_scores = (
    query_results_array.sum(
        axis=1
    )
)

exact_scores = authenticated_scores[
    0:302
]

noisy_scores = authenticated_scores[
    302:604
]

unenrolled_scores_hbf = authenticated_scores[
    604:906
]


print("=" * 80)
print("HBF AUTHENTICATED-BF SCORE DISTRIBUTIONS")
print("=" * 80)

print(
    "Maximum possible score:",
    sum(d1_BFs)
)


def describe_scores(
    name,
    scores
):

    print(
        f"\n{name}"
    )

    print(
        f"  Count  : "
        f"{len(scores)}"
    )

    print(
        f"  Mean   : "
        f"{scores.mean():.2f}"
    )

    print(
        f"  Median : "
        f"{np.median(scores):.2f}"
    )

    print(
        f"  Min    : "
        f"{scores.min()}"
    )

    print(
        f"  Max    : "
        f"{scores.max()}"
    )

    print(
        f"  Std    : "
        f"{scores.std():.2f}"
    )


describe_scores(
    "Exact enrolled",
    exact_scores
)

describe_scores(
    "1% noisy enrolled",
    noisy_scores
)

describe_scores(
    "Unenrolled",
    unenrolled_scores_hbf
)

HBF AUTHENTICATED-BF SCORE DISTRIBUTIONS
Maximum possible score: 165

Exact enrolled
  Count  : 302
  Mean   : 165.00
  Median : 165.00
  Min    : 165
  Max    : 165
  Std    : 0.00

1% noisy enrolled
  Count  : 302
  Mean   : 36.10
  Median : 36.00
  Min    : 18
  Max    : 59
  Std    : 8.24

Unenrolled
  Count  : 302
  Mean   : 21.09
  Median : 21.00
  Min    : 10
  Max    : 34
  Std    : 4.81


In [30]:
# ============================================================
# CELL 47 — Original threshold evaluation
# ============================================================

ORIGINAL_THRESHOLD = 21

y_true = np.asarray(
    [1] * 604
    +
    [0] * 302,
    dtype=np.int32
)

y_pred_original = (
    authenticated_scores
    >
    ORIGINAL_THRESHOLD
).astype(
    np.int32
)


TP = int(
    np.sum(
        (y_true == 1)
        &
        (y_pred_original == 1)
    )
)

FN = int(
    np.sum(
        (y_true == 1)
        &
        (y_pred_original == 0)
    )
)

FP = int(
    np.sum(
        (y_true == 0)
        &
        (y_pred_original == 1)
    )
)

TN = int(
    np.sum(
        (y_true == 0)
        &
        (y_pred_original == 0)
    )
)


TAR = (
    TP / (TP + FN)
    if (TP + FN)
    else 0
)

FAR = (
    FP / (FP + TN)
    if (FP + TN)
    else 0
)

FRR = (
    FN / (TP + FN)
    if (TP + FN)
    else 0
)

ACC = (
    (TP + TN)
    /
    len(y_true)
)

BAL_ACC = (
    0.5
    *
    (
        TAR
        +
        (
            TN
            /
            (TN + FP)
            if (TN + FP)
            else 0
        )
    )
)


print("=" * 80)
print("ORIGINAL HBF THRESHOLD EVALUATION")
print("=" * 80)

print(
    f"Decision rule: "
    f"Member if score > "
    f"{ORIGINAL_THRESHOLD}"
)

print(
    f"\nTP: {TP}"
)

print(
    f"FN: {FN}"
)

print(
    f"FP: {FP}"
)

print(
    f"TN: {TN}"
)

print(
    f"\nTAR : "
    f"{TAR * 100:.2f}%"
)

print(
    f"FAR : "
    f"{FAR * 100:.2f}%"
)

print(
    f"FRR : "
    f"{FRR * 100:.2f}%"
)

print(
    f"ACC : "
    f"{ACC * 100:.2f}%"
)

print(
    f"BACC: "
    f"{BAL_ACC * 100:.2f}%"
)

ORIGINAL HBF THRESHOLD EVALUATION
Decision rule: Member if score > 21

TP: 597
FN: 7
FP: 134
TN: 168

TAR : 98.84%
FAR : 44.37%
FRR : 1.16%
ACC : 84.44%
BACC: 77.24%


In [31]:
# ============================================================
# CELL 48 — Complete HBF threshold sweep
# ============================================================

import numpy as np
import pandas as pd

threshold_results = []


for threshold in range(
    -1,
    166
):

    y_pred = (
        authenticated_scores
        >
        threshold
    ).astype(
        np.int32
    )


    TP = int(
        np.sum(
            (y_true == 1)
            &
            (y_pred == 1)
        )
    )

    FN = int(
        np.sum(
            (y_true == 1)
            &
            (y_pred == 0)
        )
    )

    FP = int(
        np.sum(
            (y_true == 0)
            &
            (y_pred == 1)
        )
    )

    TN = int(
        np.sum(
            (y_true == 0)
            &
            (y_pred == 0)
        )
    )


    TAR = (
        TP / (TP + FN)
    )

    FAR = (
        FP / (FP + TN)
    )

    FRR = (
        FN / (TP + FN)
    )

    TNR = (
        TN / (TN + FP)
    )

    accuracy = (
        (TP + TN)
        /
        len(y_true)
    )

    balanced_accuracy = (
        (TAR + TNR)
        /
        2
    )


    threshold_results.append(
        {
            "Threshold": threshold,
            "TP": TP,
            "FN": FN,
            "FP": FP,
            "TN": TN,
            "TAR": TAR,
            "FAR": FAR,
            "FRR": FRR,
            "Accuracy": accuracy,
            "Balanced_Accuracy":
                balanced_accuracy
        }
    )


threshold_df = pd.DataFrame(
    threshold_results
)


print(
    "✓ Evaluated",
    len(threshold_df),
    "thresholds."
)

✓ Evaluated 167 thresholds.


In [32]:
# ============================================================
# CELL 49 — Best balanced-accuracy operating point
# ============================================================

best_idx = (
    threshold_df[
        "Balanced_Accuracy"
    ].idxmax()
)

best_row = threshold_df.loc[
    best_idx
]


print("=" * 80)
print("BEST BALANCED-ACCURACY THRESHOLD")
print("=" * 80)

print(
    f"Threshold : "
    f"{int(best_row['Threshold'])}"
)

print(
    "\nDecision rule:"
)

print(
    f"Member if authenticated BF count "
    f"> {int(best_row['Threshold'])}"
)

print(
    f"\nTP : "
    f"{int(best_row['TP'])}"
)

print(
    f"FN : "
    f"{int(best_row['FN'])}"
)

print(
    f"FP : "
    f"{int(best_row['FP'])}"
)

print(
    f"TN : "
    f"{int(best_row['TN'])}"
)

print(
    f"\nTAR : "
    f"{best_row['TAR'] * 100:.2f}%"
)

print(
    f"FAR : "
    f"{best_row['FAR'] * 100:.2f}%"
)

print(
    f"FRR : "
    f"{best_row['FRR'] * 100:.2f}%"
)

print(
    f"Accuracy : "
    f"{best_row['Accuracy'] * 100:.2f}%"
)

print(
    f"Balanced Accuracy : "
    f"{best_row['Balanced_Accuracy'] * 100:.2f}%"
)

BEST BALANCED-ACCURACY THRESHOLD
Threshold : 28

Decision rule:
Member if authenticated BF count > 28

TP : 547
FN : 57
FP : 20
TN : 282

TAR : 90.56%
FAR : 6.62%
FRR : 9.44%
Accuracy : 91.50%
Balanced Accuracy : 91.97%


In [33]:
# ============================================================
# CELL 50 — Approximate Equal Error Rate
# ============================================================

eer_idx = (
    np.abs(
        threshold_df["FAR"]
        -
        threshold_df["FRR"]
    )
).idxmin()


eer_row = threshold_df.loc[
    eer_idx
]

approx_eer = (
    eer_row["FAR"]
    +
    eer_row["FRR"]
) / 2


print("=" * 80)
print("APPROXIMATE EER OPERATING POINT")
print("=" * 80)

print(
    f"Threshold : "
    f"{int(eer_row['Threshold'])}"
)

print(
    f"FAR       : "
    f"{eer_row['FAR'] * 100:.2f}%"
)

print(
    f"FRR       : "
    f"{eer_row['FRR'] * 100:.2f}%"
)

print(
    f"Approx EER: "
    f"{approx_eer * 100:.2f}%"
)

APPROXIMATE EER OPERATING POINT
Threshold : 27
FAR       : 10.26%
FRR       : 7.78%
Approx EER: 9.02%


In [34]:
# ============================================================
# CELL 51 — Inspect useful threshold range
# ============================================================

display_columns = [
    "Threshold",
    "TP",
    "FN",
    "FP",
    "TN",
    "TAR",
    "FAR",
    "FRR",
    "Accuracy",
    "Balanced_Accuracy"
]


region = threshold_df[
    (
        threshold_df["Threshold"]
        >= 20
    )
    &
    (
        threshold_df["Threshold"]
        <= 45
    )
][
    display_columns
].copy()


for col in [
    "TAR",
    "FAR",
    "FRR",
    "Accuracy",
    "Balanced_Accuracy"
]:

    region[col] = (
        region[col]
        *
        100
    ).round(2)


print("=" * 80)
print("THRESHOLD REGION 20–45")
print("=" * 80)

print(
    region.to_string(
        index=False
    )
)

THRESHOLD REGION 20–45
 Threshold  TP  FN  FP  TN   TAR   FAR   FRR  Accuracy  Balanced_Accuracy
        20 598   6 166 136 99.01 54.97  0.99     81.02              72.02
        21 597   7 134 168 98.84 44.37  1.16     84.44              77.24
        22 593  11 106 196 98.18 35.10  1.82     87.09              81.54
        23 588  16  93 209 97.35 30.79  2.65     87.97              83.28
        24 585  19  69 233 96.85 22.85  3.15     90.29              87.00
        25 578  26  53 249 95.70 17.55  4.30     91.28              89.07
        26 568  36  38 264 94.04 12.58  5.96     91.83              90.73
        27 557  47  31 271 92.22 10.26  7.78     91.39              90.98
        28 547  57  20 282 90.56  6.62  9.44     91.50              91.97
        29 538  66  18 284 89.07  5.96 10.93     90.73              91.56
        30 524  80  11 291 86.75  3.64 13.25     89.96              91.56
        31 502 102   6 296 83.11  1.99 16.89     88.08              90.56
        32 489 

In [36]:
# ============================================================
# CELL 52 — Separate exact/noisy performance
# ============================================================

best_threshold = int(
    best_row[
        "Threshold"
    ]
)


exact_accept = (
    exact_scores
    >
    best_threshold
)

noisy_accept = (
    noisy_scores
    >
    best_threshold
)

unenrolled_accept = (
    unenrolled_scores_hbf
    >
    best_threshold
)


exact_TAR = (
    exact_accept.mean()
)

noisy_TAR = (
    noisy_accept.mean()
)

FAR = (
    unenrolled_accept.mean()
)


print("=" * 80)
print("GROUP-SPECIFIC PERFORMANCE")
print("=" * 80)

print(
    f"Threshold > "
    f"{best_threshold}"
)

print(
    f"\nExact-query TAR : "
    f"{exact_TAR * 100:.2f}%"
)

print(
    f"Noisy-query TAR : "
    f"{noisy_TAR * 100:.2f}%"
)

print(
    f"Unenrolled FAR  : "
    f"{FAR * 100:.2f}%"
)

print(
    "\nAccepted:"
)

print(
    f"  Exact      : "
    f"{exact_accept.sum()}/302"
)

print(
    f"  noisy   : "
    f"{noisy_accept.sum()}/302"
)

print(
    f"  Unenrolled : "
    f"{unenrolled_accept.sum()}/302"
)

GROUP-SPECIFIC PERFORMANCE
Threshold > 28

Exact-query TAR : 100.00%
Noisy-query TAR : 81.13%
Unenrolled FAR  : 6.62%

Accepted:
  Exact      : 302/302
  noisy   : 245/302
  Unenrolled : 20/302
